### Notebook scope

### Purpose
Test Z300 stationary-wave source metrics across cases.

### Scientific question
Does tropospheric stationary-wave geometry explain EP100 anomalies outside 0008-01?

### Method
Compute Z300 stationary projection, closeness, ACC, wave amplitudes, and correlate with EP100 wave components.

### Expected output
One source metric heatmap and selected pointwise map figures.

### Interpretation
Consistent projection/W2 amplitude links to EP100 W1+W2 support a tropospheric stationary-wave source.

### Caveat
Z300 computation reads raw Z3 and may be slow the first time; caches are written under outputs/cache.


### Setup imports and shared paths

### Purpose
Load the shared Extention_analysis utility module and create output directories.

### Scientific question
Can every notebook start from a clean kernel and still find the same data roots and output roots?

### Method
Use DATA_ROOT, HINDCAST_ROOT, BWCN_ROOT, B2000_ROOT, WORK_ROOT, FIG_DIR, TAB_DIR, CACHE_DIR, and LOG_DIR from hindcast_extension_utils.py. No diagnostics are calculated in this cell.

### Expected output
Printed path check only. No figure is generated by this setup cell.

### Interpretation
Successful execution means the notebook can access the common utilities and write outputs in the agreed directory tree.

### Caveat
This setup does not prove that every downstream data product exists; missing products are logged later.


In [ ]:
from pathlib import Path
import sys

WORK_ROOT = Path("/home/weiji/restart_exam/code_cleaned/Hindcast_experiment/Extention_analysis")
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

import math
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

import hindcast_extension_utils as hx
from hindcast_extension_utils import *
_assign_member_short = hx._assign_member_short

for d in [FIG_DIR, TAB_DIR, CACHE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("WORK_ROOT", WORK_ROOT)
print("HINDCAST_ROOT", HINDCAST_ROOT)

### Compute Z300 source metrics and heatmap

### Purpose
Build scalar Z300 metrics for all source-diagnosable cases.

### Scientific question
Which Z300 source metric best explains EP100 W1+W2 or wave2?

### Method
For each case, use source window, remove zonal mean, project onto B2000 stationary target, calculate pattern correlation and Fourier amplitudes.

### Expected output
outputs/tables/05_Z300_source_metrics_all_cases.csv and outputs/figures/05_Z300_source_metric_heatmap_all_cases.png/pdf.

### Interpretation
Strong positive projection vs W1+W2 or wave2 amplitude vs wave2 supports constructive planetary-wave source geometry.

### Caveat
Later-initialized cases are source-window tests after initialization, not long-lead precursor tests.


In [ ]:
inv = discover_hindcast_cases()
rows = []
for case in inv.loc[inv["can_source_diagnose"], "case"]:
    window = source_windows_for_case(case)["primary"]
    z = load_or_build_z300(case, window)
    target = compute_z300_climatological_stationary_target(window)
    ep = compute_all_ep100(case, window)
    if z is None or target is None or ep.empty:
        continue
    for mid in z["member"].values:
        za = compute_z300_stationary_anomaly(z.sel(member=mid))
        row = {"case": case, "member": member_short_id(mid), "Z300_stationary_projection": weighted_projection(za, target), "Z300_stationary_closeness": weighted_pattern_corr(za, target)}
        row["Z300_ACC_to_reference"] = row["Z300_stationary_closeness"]
        for k in [1,2,3,4,5,6]:
            row[f"Z300_wave{k}_amplitude"] = z300_wave_amplitude(za, k)
        row["Z300_synoptic_3_6_amplitude"] = math.sqrt(sum(row[f"Z300_wave{k}_amplitude"]**2 for k in [3,4,5,6]))
        rows.append(row)
zmet = pd.DataFrame(rows)
if not zmet.empty:
    all_ep = []
    for case in zmet["case"].unique():
        all_ep.append(compute_all_ep100(case, source_windows_for_case(case)["primary"]).assign(case=case))
    zjoin = zmet.merge(pd.concat(all_ep, ignore_index=True), on=["case", "member"], how="left")
else:
    zjoin = pd.DataFrame()
z_csv = TAB_DIR / "05_Z300_source_metrics_all_cases.csv"
zjoin.to_csv(z_csv, index=False)
relationships = [("Z300_stationary_projection", "EP100_all_waves"), ("Z300_stationary_projection", "EP100_wave1_plus_wave2"), ("Z300_stationary_closeness", "EP100_wave1_plus_wave2"), ("Z300_ACC_to_reference", "EP100_wave1_plus_wave2"), ("Z300_wave2_amplitude", "EP100_wave2"), ("Z300_synoptic_3_6_amplitude", "EP100_wave_rest")]
cor_rows = []
for case, sub in zjoin.groupby("case") if not zjoin.empty else []:
    for xcol, ycol in relationships:
        c = finite_corr(sub[xcol], sub[ycol])
        cor_rows.append({"case": case, "relationship": f"{xcol} vs {ycol}", **c})
cor = pd.DataFrame(cor_rows)
cor_csv = TAB_DIR / "05_Z300_source_metric_correlations_all_cases.csv"
cor.to_csv(cor_csv, index=False)
mat = cor.pivot(index="case", columns="relationship", values="R") if not cor.empty else pd.DataFrame()
fig, ax = plt.subplots(figsize=(14, max(4, 0.35*len(mat)+2)), constrained_layout=True)
if mat.empty:
    ax.axis("off"); ax.text(0.5, 0.5, "No Z300 source metrics", ha="center", va="center")
else:
    im = ax.imshow(mat.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(mat.shape[1]), mat.columns, rotation=45, ha="right")
    ax.set_yticks(range(mat.shape[0]), mat.index)
    for i, case in enumerate(mat.index):
        for j, col in enumerate(mat.columns):
            p = cor[(cor["case"]==case)&(cor["relationship"]==col)]["p"]
            star = "*" if len(p) and p.iloc[0] < 0.05 else ""
            ax.text(j, i, f"{mat.iloc[i,j]:.2f}{star}" if np.isfinite(mat.iloc[i,j]) else "NA", ha="center", va="center", fontsize=7)
    fig.colorbar(im, ax=ax, label="R")
    ax.set_title("Z300 source metric heatmap")
savefig(fig, "05_Z300_source_metric_heatmap_all_cases", notebook="05_Z300_stationary_source_multicase.ipynb", scientific_question="Do Z300 stationary-wave metrics explain EP100 anomalies across cases?", variables_windows="Z300 300 hPa 20-90N; EP100 100 hPa 40-80N; source windows", interpretation="Starred projection/W2 relationships support a stationary-wave source of stratospheric EP flux anomalies.", caveat="Z300 raw-data interpolation is cached but can be unavailable for incomplete cases.", csv_table=cor_csv)
plt.show()

### Selected pointwise correlation maps

### Purpose
Map local Z300 anomaly correlations with EP and O3 metrics for representative cases.

### Scientific question
Does the spatial pattern of correlations align with climatological stationary wave ridges/troughs?

### Method
For selected cases, correlate member Z300 stationary anomaly at each grid point with EP100 all/wave1/wave2/W1+W2/rest and O3 RMSE. Contours are B2000 stationary-wave target.

### Expected output
outputs/figures/05_Z300_pointwise_corr_selected_<case>.png/pdf and outputs/tables/05_Z300_pointwise_corr_selected_summary.csv.

### Interpretation
Spatially coherent positive/negative lobes aligned with stationary-wave contours support constructive/destructive interference.

### Caveat
Pointwise p-values are not field-significance corrected; use as source-hunting maps.


In [ ]:
summary_rows = []
selected_specs = [("0008-01", "Jan", MONTH_WINDOWS["Jan"]), ("0008-01", "Feb", MONTH_WINDOWS["Feb"])]
if not cor.empty:
    extra_cases = cor.loc[cor["relationship"].str.contains("stationary_projection")].sort_values("R", ascending=False)["case"].drop_duplicates().tolist()
    for c in extra_cases:
        if c != "0008-01" and len(selected_specs) < 4:
            selected_specs.append((c, "primary", source_windows_for_case(c)["primary"]))
# Add one NOCOUPL case if available.
for c in discover_hindcast_cases().query("config == 'NOCOUPL'")["case"].tolist():
    if all(c != spec[0] for spec in selected_specs):
        selected_specs.append((c, "primary", source_windows_for_case(c)["primary"]))
        break

for case, label, window in selected_specs:
    z = load_or_build_z300(case, window)
    if z is None:
        continue
    za = compute_z300_stationary_anomaly(z)
    ep = compute_all_ep100(case, window)
    o3, date = load_hindcast_o3(case)
    ref, ref_date = load_bwcn_reference_o3(parse_case_name(case)["year"])
    rmse = compute_o3_rmse(o3, ref, *target_window_for_case(case))
    metrics = ep.merge(rmse, on="member", how="outer") if not ep.empty else rmse.copy()
    if metrics.empty or "member" not in metrics:
        log_message(f"{case} {label}: no member metrics for selected Z300 map")
        continue
    targets = ["EP100_all_waves", "EP100_wave1", "EP100_wave2", "EP100_wave1_plus_wave2", "EP100_wave_rest", "O3_RMSE"]
    target = compute_z300_climatological_stationary_target(window)
    fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
    im = None
    case_rows = []
    for ax, col in zip(axes.ravel(), targets):
        ds = pointwise_member_correlation(za, metrics.set_index("member")[col]) if col in metrics else None
        if ds is None:
            ax.axis("off"); ax.set_title(f"{col}: missing")
            continue
        im = ax.contourf(ds["lon"], ds["lat"], ds["R"], levels=np.linspace(-0.8,0.8,17), cmap="RdBu_r", extend="both")
        if target is not None:
            ax.contour(target["lon"], target["lat"], target, levels=np.arange(-240, 241, 40), colors="k", linewidths=0.45)
        ax.contour(ds["lon"], ds["lat"], ds["p"] < 0.05, levels=[0.5], colors="0.1", linewidths=0.35)
        ax.set_title(col, fontsize=8); ax.set_xlabel("lon"); ax.set_ylabel("lat")
        rec = {"case": case, "window_label": label, "metric": col, "mean_abs_R": float(np.nanmean(np.abs(ds["R"].values)))}
        summary_rows.append(rec); case_rows.append(rec)
    if im is not None:
        fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.75, label="pointwise R")
    fig.suptitle(f"{case} {label}: Z300 pointwise member correlations; contours = B2000 stationary waves")
    safe_label = str(label).replace("/", "_")
    case_csv = TAB_DIR / f"05_Z300_pointwise_corr_selected_{case}_{safe_label}_summary.csv"
    pd.DataFrame(case_rows).to_csv(case_csv, index=False)
    savefig(fig, f"05_Z300_pointwise_corr_selected_{case}_{safe_label}", notebook="05_Z300_stationary_source_multicase.ipynb", scientific_question="Where does Z300 anomaly correlate with EP/O3 metrics for selected cases and windows?", variables_windows="Z300 300 hPa stationary anomaly; EP100 components; O3 RMSE; selected Jan/Feb/primary source windows", interpretation="Coherent lobes near stationary-wave contours indicate possible constructive/destructive interference source regions.", caveat="These pointwise maps are exploratory and not multiple-comparison corrected.", csv_table=case_csv)
    plt.close(fig)
summary = pd.DataFrame(summary_rows)
summary.to_csv(TAB_DIR / "05_Z300_pointwise_corr_selected_summary.csv", index=False)
write_figure_guide()
